In [ ]:
%load_ext dotenv
%dotenv
    

In [ ]:
import pandas as pd
import os
import pinecone import Pinecone , ServerlessSpec
from dotenv import laod_env , find_dotenv
import pinecone
from sentences_transformers import Sentencetransformer


In [ ]:
files = pd.read_csv("course_section.csv" , encoding = "ANSI")

In [ ]:
files["unique_id"] = files["course_id"].astype(str) + "-" + files["section_id"].astype(str)

In [ ]:
files["metadata"] = files.apply(lambda row: {
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
    
} , axis = 1)

In [ ]:
def create_embadding(row):
    combined_text = f'''{row['course_name']} {row["course_technology"]}
                        {row['course_description']} {row['section_name']}{row['section_description']}'''
    return model.encode(combined_text , show_progress_bar = False)

In [ ]:
model = Sentencetransformers('all-MiniLM-L6-v2')

In [ ]:
flies['embadding'] = files.apply(create_embedding , axis = 1)

In [ ]:
## upserting data to pinecone

In [ ]:
load_dotenv(find_dotenv() , override = True)

In [ ]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY") , environment = os.environ.get("PINECONE_ENV"))

In [ ]:
index_name = "my_index"
dimension = 384
metric = "cosine"

In [ ]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")
else:
    print(f"{index_name} not in index_list")

In [ ]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
        
        
    )
)

In [ ]:
index = pc.Index(name = index_name)

In [ ]:
vector_to_upsert = [(row["unique_id"] , row["embadding"].tolist(),  row["metadata"]) for index , row in files.iterrows()]

In [ ]:
index.upsert(vectors = vectors_to_upsert)
print("data successfully upserted to pinecone index")